# SPAM EMAIL DETECTOR - NEUROSENTRY

**Muhammad Ahmed FA22-BSCS-0194**

---


**Yamaan Khan FA23-BSCS-0162**

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)


In [ ]:
data=pd.read_csv('spam.csv')
data

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [ ]:
data.columns

Index(['Category', 'Message'], dtype='object')

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Category  5572 non-null   object
 1   Message   5572 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


**Dropped The Column Unnamed: 0**

In [ ]:
data.isna().sum()

,0
Category,0
Message,0


In [ ]:
print(data.isnull().sum())
print(data['Spam'].value_counts())

Category    0
Message     0
Spam        0
dtype: int64
Spam
0    4825
1     747
Name: count, dtype: int64


In [ ]:
data['Spam']=data['Category'].apply(lambda x:1 if x=='spam' else 0)
data.head(5)

,Category,Message,Spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

X = data['Message']
y = data['Spam']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline_tfidf = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", LogisticRegression(class_weight="balanced"))
])

pipeline_tfidf.fit(X_train, y_train)

Pipeline(steps=[('tfidf', TfidfVectorizer()),
                ('model', LogisticRegression(class_weight='balanced'))])

In [ ]:
y_pred_default = pipeline_tfidf.predict(X_test)
print("=== Default Threshold (0.5) ===")
print(classification_report(y_test, y_pred_default))

=== Default Threshold (0.5) ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       966
           1       0.95      0.93      0.94       149

    accuracy                           0.98      1115
   macro avg       0.97      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
probs = pipeline_tfidf.predict_proba(X_test)[:,1]

# change threshold here
threshold = 0.4

y_pred_custom = (probs > threshold).astype(int)

print("=== Custom Threshold (0.4) ===")
print(classification_report(y_test, y_pred_custom))

=== Custom Threshold (0.4) ===
              precision    recall  f1-score   support

           0       0.99      0.98      0.99       966
           1       0.87      0.96      0.91       149

    accuracy                           0.97      1115
   macro avg       0.93      0.97      0.95      1115
weighted avg       0.98      0.97      0.98      1115



🧪 EXPERIMENT 2: Naive Bayes + CountVectorizer

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

pipeline_nb = Pipeline([
    ("count", CountVectorizer()),
    ("model", MultinomialNB())
])

pipeline_nb.fit(X_train, y_train)

y_pred_nb = pipeline_nb.predict(X_test)

print("=== Naive Bayes + CountVectorizer ===")
print(classification_report(y_test, y_pred_nb))

=== Naive Bayes + CountVectorizer ===
              precision    recall  f1-score   support

           0       0.99      1.00      0.99       966
           1       0.99      0.91      0.95       149

    accuracy                           0.99      1115
   macro avg       0.99      0.96      0.97      1115
weighted avg       0.99      0.99      0.99      1115



🧪 EXPERIMENT 3: TF-IDF with N-grams

In [ ]:
pipeline_ngram = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1,2))),
    ("model", LogisticRegression(class_weight="balanced"))
])

pipeline_ngram.fit(X_train, y_train)

y_pred_ngram = pipeline_ngram.predict(X_test)

print("=== TF-IDF + N-grams ===")
print(classification_report(y_test, y_pred_ngram))

=== TF-IDF + N-grams ===
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       966
           1       0.94      0.93      0.93       149

    accuracy                           0.98      1115
   macro avg       0.96      0.96      0.96      1115
weighted avg       0.98      0.98      0.98      1115



# Tarining The Model

**Here I given Two email Two detect 1st One is looking good and the other one looking spam**

In [ ]:
final_pipeline=pipeline_ngram

test_cases = [
    "Free entry in a contest!!!",
    "Call me when you reach home",
    "URGENT! You have won 5000 dollars",
    "kal uni aa raha hai?",
    "click this link to claim reward",
    "bhai scene kya hai aj?"
]

for msg in test_cases:
    prob = final_pipeline.predict_proba([msg])[0][1]
    pred = "SPAM" if prob > 0.4 else "NOT SPAM"
    print(msg)
    print("Prediction:", pred, "| Prob:", prob)
    print("-"*50)

print("final_pipeline=pipeline_ngram")

Free entry in a contest!!!
Prediction: SPAM | Prob: 0.6032367104553715
--------------------------------------------------
Call me when you reach home
Prediction: NOT SPAM | Prob: 0.08989199330979894
--------------------------------------------------
URGENT! You have won 5000 dollars
Prediction: SPAM | Prob: 0.7549973653322836
--------------------------------------------------
kal uni aa raha hai?
Prediction: NOT SPAM | Prob: 0.1846307014393693
--------------------------------------------------
click this link to claim reward
Prediction: SPAM | Prob: 0.7813863597608032
--------------------------------------------------
bhai scene kya hai aj?
Prediction: NOT SPAM | Prob: 0.18783777950249053
--------------------------------------------------
final_pipeline=pipeline_ngram


**Predict Email**

In [ ]:
final_pipeline = pipeline_ngram
threshold = 0.4

import pickle

artifact = {
    "pipeline": final_pipeline,
    "threshold": threshold
}

with open("spam_detector.pkl", "wb") as f:
    pickle.dump(artifact, f)

In [ ]:
import pickle

with open("spam_detector.pkl", "rb") as f:
    artifact = pickle.load(f)

pipeline = artifact["pipeline"]
threshold = artifact["threshold"]

In [ ]:
def predict_spam(text):
    prob = pipeline.predict_proba([text])[0][1]
    pred = "SPAM" if prob > threshold else "NOT SPAM"
    return pred, prob

In [ ]:
emails = [
    "Free entry in a contest!!!",
    "Call me when you reach home",
    "bhai scene kya hai aj?",
    "click this link to claim reward"
]

for msg in emails:
    pred, prob = predict_spam(msg)
    print(msg)
    print("Prediction:", pred, "| Prob:", prob)
    print("-"*50)

Free entry in a contest!!!
Prediction: SPAM | Prob: 0.6032367104553715
--------------------------------------------------
Call me when you reach home
Prediction: NOT SPAM | Prob: 0.08989199330979894
--------------------------------------------------
bhai scene kya hai aj?
Prediction: NOT SPAM | Prob: 0.18783777950249053
--------------------------------------------------
click this link to claim reward
Prediction: SPAM | Prob: 0.7813863597608032
--------------------------------------------------


In [ ]:
import pickle

artifact = {
    "pipeline": final_pipeline,
    "threshold": threshold
}

with open("spam_detector.pkl", "wb") as f:
    pickle.dump(artifact, f)